# 04 — Parallel Infrastructure Benchmark
9 runs (N ∈ {1,2,4} × corpus ∈ {1k,5k,20k}). Generates Amdahl curve + latency plot.

In [1]:
%load_ext autoreload
%autoreload 2

import sys, os
sys.path.insert(0, os.path.abspath('..'))
os.chdir('..')

In [2]:
import json
import pandas as pd
from config import CORPUS_DIR, LOG_DIR, ACCURACY_DIR

# Load pre-built corpora from notebook 01
corpora = {}
for size in [1000, 5000, 20000]:
    path = os.path.join(CORPUS_DIR, f'corpus_{size}.json')
    with open(path) as f:
        corpora[size] = json.load(f)
    print(f'Loaded corpus_{size}: {len(corpora[size])} docs')

# Flatten to a single list ordered by size (benchmark slices by corpus_size)
all_docs = corpora[20000]  # largest; benchmark trims to corpus_size
print(f'Total docs available for benchmarking: {len(all_docs)}')

Loaded corpus_1000: 1000 docs
Loaded corpus_5000: 5000 docs
Loaded corpus_20000: 6251 docs
Total docs available for benchmarking: 6251


## Run Benchmark (9 runs)

In [3]:
from parallel.benchmark import run_full_benchmark
from config import CORPUS_SIZES, PROCESS_COUNTS

# NOTE: N=1 runs first per corpus size — this is T_sequential for Amdahl
df_bench = run_full_benchmark(all_docs, corpus_sizes=CORPUS_SIZES, process_counts=PROCESS_COUNTS)
print(df_bench)

Running benchmark: corpus_size=1000, n_processes=1


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  total_s=69.56  chunks=2658
Running benchmark: corpus_size=1000, n_processes=2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9524.31it/s]


  total_s=71.48  chunks=2658
Running benchmark: corpus_size=1000, n_processes=4


...

  total_s=452.41  chunks=17013
Running benchmark: corpus_size=20000, n_processes=2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10799.52it/s]


  total_s=436.06  chunks=17013
Running benchmark: corpus_size=20000, n_processes=4


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13651.00it/s]


  total_s=496.03  chunks=17013
   chunking_s  embedding_s  indexing_s     total_s  n_processes  corpus_size  \
0    0.439443    69.116037    0.000810   69.556290            1         1000   
1    0.400151    70.828220    0.248296   71.476667            2         1000   
2    0.440551    83.553765    0.237289   84.231605            4         1000   
3    1.793367   358.685511    0.013217  360.492095            1         5000   
4    1.550762   358.523991    0.288764  360.363516            2         5000   
5    1.305194   393.881644    0.393177  395.580015            4         5000   
6    2.434671   449.972733    0.005329  452.412732            1        20000   
7    1.786713   433.989491    0.288499  436.064703            2        20000   
8    1.364267   494.274093    0.387942  496.026303            4        20000   

   n_chunks  
0      2658  
1      2658  
2      2658  
3     13497  
4     13497  
5     13497  
6     17013  
7     17013  
8     17013  


## Speedup Analysis

In [4]:
from evaluation.amdahl import compute_speedup, fit_sequential_fraction

df_speedup = compute_speedup(df_bench)
S_fitted = fit_sequential_fraction(df_speedup)
print(f'Fitted sequential fraction S = {S_fitted:.3f}')
print(df_speedup[['corpus_size', 'n_processes', 'total_s', 'speedup_empirical']])

Fitted sequential fraction S = 0.990
   corpus_size  n_processes     total_s  speedup_empirical
0         1000            1   69.556290           1.000000
1         1000            2   71.476667           0.973133
2         1000            4   84.231605           0.825774
3         5000            1  360.492095           1.000000
4         5000            2  360.363516           1.000357
5         5000            4  395.580015           0.911300
6        20000            1  452.412732           1.000000
7        20000            2  436.064703           1.037490
8        20000            4  496.026303           0.912074


## Generate Plots

In [5]:
from evaluation.reporter import plot_amdahl, plot_latency_vs_corpus, plot_accuracy_comparison, print_results_table

plot_amdahl(df_bench)
plot_latency_vs_corpus(df_bench)

Saved: /home/zaid/Desktop/FAST/NLP/Project/3/multihop-rag-pdc/results/plots/amdahl_speedup.png
Saved: /home/zaid/Desktop/FAST/NLP/Project/3/multihop-rag-pdc/results/plots/latency_vs_corpus.png


In [6]:
# Load accuracy results from notebooks 02 and 03
with open(os.path.join(ACCURACY_DIR, 'baseline_results.json')) as f:
    baseline_acc = json.load(f)
with open(os.path.join(ACCURACY_DIR, 'multihop_results.json')) as f:
    multihop_acc = json.load(f)

plot_accuracy_comparison(baseline_acc, multihop_acc)
print_results_table(baseline_acc, multihop_acc, df_bench)

Saved: /home/zaid/Desktop/FAST/NLP/Project/3/multihop-rag-pdc/results/plots/accuracy_comparison.png

=== Final Results Table ===
Phase                                           FinQA EM  MultiHop F1   T(N=1)   T(N=max)  Speedup
-----------------------------------------------------------------------------------------------
Phase 1 — Sequential Baseline                      0.000        0.008   360.5s          —       1×
Phase 2 — Multi-Hop (Sequential Infra)             0.000        0.099   360.5s          —       1×
Phase 3 — Multi-Hop (Parallel Infra)               0.000        0.099        —     395.6s    0.91×
